In [ ]:
import os
import yaml
from collections import defaultdict
from tqdm import tqdm

# Base path
base_path = r"F:\0Defense\Dataset\Vegetables_Dataset_seperate_folder"

splits_map = {
    "train": "train",
    "val": "valid",
    "test": "test"
}

vegetable_folders = [
    d for d in os.listdir(base_path)
    if os.path.isdir(os.path.join(base_path, d))
]

for veg in vegetable_folders:

    print("\n" + "="*100)
    print(f"Processing Vegetable: {veg}")
    print("="*100)

    yaml_path = os.path.join(base_path, veg, "data.yaml")

    if not os.path.exists(yaml_path):
        print("No data.yaml found, skipping...")
        continue

    # Read YAML
    with open(yaml_path, 'r') as f:
        data_yaml = yaml.safe_load(f)

    class_names = data_yaml['names']
    num_classes = data_yaml['nc']

    print(f"Number of Classes: {num_classes}")
    print(f"Classes: {class_names}")

    # Counters
    total_image_count = defaultdict(int)
    total_annotation_count = defaultdict(int)

    split_image_count = {s: defaultdict(int) for s in splits_map}
    split_annotation_count = {s: defaultdict(int) for s in splits_map}

    # Loop through splits
    for yaml_split, folder_name in splits_map.items():

        labels_path = os.path.join(base_path, veg, folder_name, "labels")
        images_path = os.path.join(base_path, veg, folder_name, "images")

        if not os.path.exists(labels_path):
            continue

        label_files = [f for f in os.listdir(labels_path) if f.endswith(".txt")]

        for file in tqdm(label_files, desc=f"{veg}-{folder_name}", unit="file"):

            label_file = os.path.join(labels_path, file)
            image_file_jpg = os.path.join(images_path, file.replace(".txt", ".jpg"))
            image_file_png = os.path.join(images_path, file.replace(".txt", ".png"))

            if os.path.exists(image_file_jpg):
                image_file = image_file_jpg
            elif os.path.exists(image_file_png):
                image_file = image_file_png
            else:
                continue

            with open(label_file, "r") as f:
                lines = f.readlines()

            used_classes = set()

            for line in lines:
                class_id = int(line.split()[0])

                split_annotation_count[yaml_split][class_id] += 1
                total_annotation_count[class_id] += 1

                used_classes.add(class_id)

            for cid in used_classes:
                split_image_count[yaml_split][cid] += 1
                total_image_count[cid] += 1

    # ===============================
    # TABLE 1 — OVERALL
    # ===============================
    print("\nOVERALL STATISTICS")
    print("-"*90)
    print(f"{'Class':30} {'Images':>10} {'Annotations':>15}")
    print("-"*90)

    for i, name in enumerate(class_names):
        print(f"{name:30} {total_image_count[i]:10} {total_annotation_count[i]:15}")

    # ===============================
    # TABLE 2 — SPLIT-WISE
    # ===============================
    print("\nSPLIT-WISE STATISTICS")
    print("-"*120)
    print(f"{'Class':25} | {'Train Img':>9} {'Train Ann':>9} | {'Valid Img':>9} {'Valid Ann':>9} | {'Test Img':>9} {'Test Ann':>9}")
    print("-"*120)

    for i, name in enumerate(class_names):
        print(
            f"{name:25} | "
            f"{split_image_count['train'][i]:9} {split_annotation_count['train'][i]:9} | "
            f"{split_image_count['val'][i]:9} {split_annotation_count['val'][i]:9} | "
            f"{split_image_count['test'][i]:9} {split_annotation_count['test'][i]:9}"
        )